In [ ]:
# Optuna 하이퍼파라미터 자동 최적화 (ResNet34 Transfer Learning)

## Step 1. 라이브러리 임포트
# PyTorch, torchvision, numpy, optuna 등 전체 파이프라인에서 사용할 라이브러리를 불러온다.


In [ ]:
# 딥러닝, 데이터 처리, 하이퍼파라미터 최적화에 필요한 라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import numpy as np
import optuna  # 하이퍼파라미터 자동 최적화 프레임워크


## Step 2. 데이터 전처리 (Transform 정의)

학습/검증 이미지에 적용할 전처리 파이프라인을 정의한다.

- `transform_train`: 224×224 리사이즈 + 좌우반전(데이터 증강) + 정규화
- `transform_test`: 증강 없이 리사이즈 + 정규화만 적용


In [ ]:
# 학습용 transform: 224x224 리사이즈 + 좌우반전(데이터 증강) + 정규화
transform_train = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

# 검증/테스트용 transform: 증강 없이 리사이즈 + 정규화만 적용
transform_test = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


## Step 3. 데이터셋 로드

`ImageFolder`를 사용해 폴더 구조에서 자동으로 클래스 레이블을 감지하고 데이터셋을 생성한다.  
폴더 구조: `dataset/train/<클래스명>/이미지파일`


In [25]:
train_datasets = datasets.ImageFolder(
    root="../resnet/dataset/train", transform=transform_train
)
test_datasets = datasets.ImageFolder(
    root="../resnet/dataset/test", transform=transform_test
)

## Step 4. Train / Validation 데이터 분할

전체 학습 데이터를 **8:2 비율**로 train / validation 세트로 분리한다.  
`random_split`을 사용하므로 매 실행마다 무작위 분할이 이루어진다.

> DataLoader는 `batch_size`가 Optuna 탐색 대상이므로 `objective` 함수 내부에서 생성한다.


In [26]:
from torch.utils.data import random_split

train_data_size = len(train_datasets)

# 8:2  train:8   test:2
train_size = int(train_data_size * 0.8)
val_size = train_data_size - train_size

print(f"train data size : {train_size}    val data size : {val_size}")

train_dataset, val_dataset = random_split(train_datasets, [train_size, val_size])

# batchsize 최적화를 위해 objective 함수 안에서 처리
# train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=True)
# val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=4, shuffle=True)

train data size : 96    val data size : 24


## Step 5. 디바이스 및 모델 설정

- GPU/CPU 디바이스를 확인하고 모델을 해당 디바이스로 이동한다.
- ResNet34의 ImageNet 사전학습 가중치를 로드한 뒤, 마지막 FC 레이어를 **3-class 분류**용으로 교체한다 (Transfer Learning).


In [27]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = models.resnet34(pretrained=True)
model.fc = nn.Linear(512, 3)
model = model.to(device)

c:\ml_proj4\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\ml_proj4\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## Step 6. Optuna Objective 함수 정의 및 최적화 실행

`objective(trial)` 함수는 Optuna가 매 Trial마다 호출하는 핵심 함수다.

| 탐색 파라미터 | 범위        | 설명                      |
| ------------- | ----------- | ------------------------- |
| `batch_size`  | 4, 6, 8     | 미니배치 크기             |
| `lr`          | 1e-5 ~ 1e-3 | Adam 학습률 (로그 스케일) |

- **학습 루프**: 순전파 → 손실 계산 → 역전파 → 가중치 업데이트
- **검증 루프**: gradient 비활성화 상태에서 val loss 누적
- **Pruning**: `trial.should_prune()`으로 성능이 나쁜 Trial을 조기 종료하여 탐색 효율 향상


In [28]:
def objective(trial):

    batch_size = trial.suggest_categorical("batch_size", [4, 6, 8])
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-3)

    train_dataloader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, shuffle=True
    )

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    epochs = 20
    val_loss = 0

    for epoch in range(epochs):
        model.train()
        print(epoch)

        for img, labels in train_dataloader:
            optimizer.zero_grad()
            preds = model(img.to(device))
            loss = criterion(preds, labels.to(device))
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            for img, labels in val_dataloader:
                img = img.to(device)
                labels = labels.to(device)
                preds = model(img)
                val_loss += criterion(preds, labels)

        total_loss = val_loss / len(val_dataloader)

        trial.report(total_loss, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return total_loss


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)


[I 2026-03-06 10:58:04,874] A new study created in memory with name: no-name-a1627291-ec42-4624-9f76-bdcd1d482fe6
C:\Users\user\AppData\Local\Temp\ipykernel_47300\1506979331.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-5, 1e-3)


0
1
2
3
4


[W 2026-03-06 10:59:20,217] Trial 0 failed with parameters: {'batch_size': 4, 'lr': 1.0451881645366466e-05} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\ml_proj4\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_47300\1506979331.py", line 26, in objective
    loss.backward()
    ~~~~~~~~~~~~~^^
  File "c:\ml_proj4\.venv\Lib\site-packages\torch\_tensor.py", line 630, in backward
    torch.autograd.backward(
    ~~~~~~~~~~~~~~~~~~~~~~~^
        self, gradient, retain_graph, create_graph, inputs=inputs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\ml_proj4\.venv\Lib\site-packages\torch\autograd\__init__.py", line 364, in backward
    _engine_run_backward(
    ~~~~~~~~~~~~~~~~~~~~^
        tensors,
        ^^^^^^^^
    ...<5 lines>...
        accumulate_grad=True,
        ^^^^^^^^^^^^^

KeyboardInterrupt: 

## Step 7. 최적 하이퍼파라미터 출력

50번의 Trial 중 **validation loss가 가장 낮았던 Trial**의 하이퍼파라미터를 출력한다.  
`study.best_trial.params` 딕셔너리에 최적 `batch_size`와 `lr` 값이 담겨 있다.


In [ ]:
print(study.best_trial.params)

{'batch_size': 4, 'lr': 0.00013573022624209873}
